In [1]:
import pandas as pd
import joblib
import numpy as np 
# Load model
xgb_model = joblib.load("xgb_model.pkl")

# Load feature columns
feature_columns = joblib.load("feature_columns.pkl")

df = pd.read_csv("odi_final_dataset.csv")

In [2]:
df.head()

,date,team1,team2,city,venue,toss_winner,toss_decision,winner,year,home_team,team1_recent_form,team2_recent_form,team1_h2h_wins,team2_h2h_wins,toss_match_win,target,team1_strength,team2_strength
0,2015-01-11,New Zealand,Sri Lanka,Christchurch,Hagley Oval,Sri Lanka,bat,New Zealand,2015,New Zealand,0,0,0,0,0,1,0.627119,0.418994
1,2015-01-12,Ireland,Scotland,Unknown,Dubai International Cricket Stadium,Ireland,field,Ireland,2015,Neutral,0,0,0,0,1,1,0.402439,0.571429
2,2015-01-15,New Zealand,Sri Lanka,Hamilton,Seddon Park,New Zealand,bat,Sri Lanka,2015,New Zealand,1,0,1,0,0,0,0.627119,0.418994
3,2015-01-16,South Africa,West Indies,Durban,Kingsmead,South Africa,bat,South Africa,2015,South Africa,0,0,0,0,1,1,0.582822,0.345912
4,2015-01-16,Australia,England,Unknown,Sydney Cricket Ground,England,bat,Australia,2015,Australia,0,0,0,0,0,1,0.570588,0.589595


In [3]:

def recent_form(team):

    matches = df[
        (df["team1"] == team) |
        (df["team2"] == team)
    ].tail(5)

    return (matches["winner"] == team).sum()

In [4]:
recent_form("India")

np.int64(2)

In [5]:
def h2h(team1, team2):

    matches = df[
        (
            (df["team1"] == team1) &
            (df["team2"] == team2)
        ) |
        (
            (df["team1"] == team2) &
            (df["team2"] == team1)
        )
    ]

    team1_wins = (matches["winner"] == team1).sum()
    team2_wins = (matches["winner"] == team2).sum()

    return team1_wins, team2_wins

In [6]:
team_strength = {}

teams = set(df["team1"]).union(set(df["team2"]))

for team in teams:

    matches = df[
        (df["team1"] == team) |
        (df["team2"] == team)
    ]

    wins = (matches["winner"] == team).sum()

    team_strength[team] = wins / len(matches)

In [7]:
team_strength["India"]

np.float64(0.673469387755102)

In [8]:
recent_form("India")

h2h("India","Australia")

team_strength["India"]

np.float64(0.673469387755102)

In [9]:
feature_columns[:50]

['team1_recent_form',
 'team2_recent_form',
 'team1_h2h_wins',
 'team2_h2h_wins',
 'team1_strength',
 'team2_strength',
 'toss_match_win',
 'team1_Bangladesh',
 'team1_Canada',
 'team1_England',
 'team1_Hong Kong',
 'team1_India',
 'team1_Ireland',
 'team1_Jersey',
 'team1_Namibia',
 'team1_Nepal',
 'team1_Netherlands',
 'team1_New Zealand',
 'team1_Oman',
 'team1_Pakistan',
 'team1_Papua New Guinea',
 'team1_Scotland',
 'team1_South Africa',
 'team1_Sri Lanka',
 'team1_United Arab Emirates',
 'team1_United States of America',
 'team1_West Indies',
 'team1_Zimbabwe',
 'team2_Bangladesh',
 'team2_Canada',
 'team2_England',
 'team2_Hong Kong',
 'team2_India',
 'team2_Ireland',
 'team2_Jersey',
 'team2_Namibia',
 'team2_Nepal',
 'team2_Netherlands',
 'team2_New Zealand',
 'team2_Oman',
 'team2_Pakistan',
 'team2_Papua New Guinea',
 'team2_Scotland',
 'team2_South Africa',
 'team2_Sri Lanka',
 'team2_United Arab Emirates',
 'team2_United States of America',
 'team2_West Indies',
 'team2_Zi

In [10]:
[c for c in feature_columns if c.startswith("venue_")][:20]

['venue_Affies Park',
 'venue_Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)',
 'venue_Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)',
 'venue_Amini Park, Port Moresby',
 'venue_Arun Jaitley Stadium',
 'venue_Arun Jaitley Stadium, Delhi',
 'venue_Barabati Stadium',
 'venue_Barabati Stadium, Cuttack',
 'venue_Barsapara Cricket Stadium',
 'venue_Barsapara Cricket Stadium, Guwahati',
 'venue_Basin Reserve',
 'venue_Basin Reserve, Wellington',
 'venue_Bay Oval',
 'venue_Bay Oval, Mount Maunganui',
 'venue_Bellerive Oval',
 'venue_Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
 'venue_Bir Sreshtho Flight Lieutenant Matiur Rahman Stadium, Chattogram',
 'venue_Boland Park',
 'venue_Boland Park, Paarl',
 'venue_Brabourne Stadium']

In [11]:
[c for c in feature_columns if "toss" in c]

['toss_match_win',
 'toss_winner_Bangladesh',
 'toss_winner_Canada',
 'toss_winner_England',
 'toss_winner_Hong Kong',
 'toss_winner_India',
 'toss_winner_Ireland',
 'toss_winner_Jersey',
 'toss_winner_Namibia',
 'toss_winner_Nepal',
 'toss_winner_Netherlands',
 'toss_winner_New Zealand',
 'toss_winner_Oman',
 'toss_winner_Pakistan',
 'toss_winner_Papua New Guinea',
 'toss_winner_Scotland',
 'toss_winner_South Africa',
 'toss_winner_Sri Lanka',
 'toss_winner_United Arab Emirates',
 'toss_winner_United States of America',
 'toss_winner_West Indies',
 'toss_winner_Zimbabwe',
 'toss_decision_field']

In [12]:
def predict_match(
    team1,
    team2,
    venue,
    toss_winner,
    toss_decision,
    home_team="Neutral"
):

    X_pred = pd.DataFrame(
        np.zeros((1, len(feature_columns))),
        columns=feature_columns
    )

    # --------------------
    # Numerical Features
    # --------------------

    X_pred["team1_recent_form"] = recent_form(team1)
    X_pred["team2_recent_form"] = recent_form(team2)

    h2h1, h2h2 = h2h(team1, team2)

    X_pred["team1_h2h_wins"] = h2h1
    X_pred["team2_h2h_wins"] = h2h2

    X_pred["team1_strength"] = team_strength.get(team1, 0.5)
    X_pred["team2_strength"] = team_strength.get(team2, 0.5)

    X_pred["toss_match_win"] = 1

    # --------------------
    # One Hot Encoding
    # --------------------

    cols = [
        f"team1_{team1}",
        f"team2_{team2}",
        f"venue_{venue}",
        f"toss_winner_{toss_winner}",
        f"toss_decision_{toss_decision}",
        f"home_team_{home_team}"
    ]

    for col in cols:

        if col in X_pred.columns:
            X_pred[col] = 1

    pred =xgb_model.predict(X_pred)[0]

    probs = xgb_model.predict_proba(X_pred)[0]

    winner = team1 if pred == 1 else team2

    return {
    "Predicted Winner": winner,
    f"{team1} Win %": float(round(probs[1] * 100, 2)),
    f"{team2} Win %": float(round(probs[0] * 100, 2))
}

In [13]:
predict_match(
    team1="India",
    team2="New Zealand",
    venue="Bay Oval",
    toss_winner="India",
    toss_decision="field",
    home_team="Neutral"
)

{'Predicted Winner': 'New Zealand',
 'India Win %': 6.889999866485596,
 'New Zealand Win %': 93.11000061035156}